# 07 - Synthetic Vegetation-Index Composite

**What:** Aggregate a vegetation-index (e.g. NDVI) image collection over time into a *single* `ee.Image` using a chosen temporal metric: **Mean, Median, Min, Max, Amplitude (max-min), Standard Deviation, Sum, Area Under Curve (AUC)**.

**Why:** This mirrors the SAR composite workflow but operates on optical vegetation indices. A composite collapses a noisy multi-date time series into one interpretable layer summarizing the growing season (mean condition, variability, total accumulated greenness via AUC, etc.).

**Legacy reference:** `legacy:ravi_dialog.py`
- `composite_clicked` / `composite` (~2899-3000) - drives the workflow from the `metrica` combobox + `indice_composicao` combobox.
- `aggregate_index_collection` (~2370-2406) - maps each metric label to its exact Earth Engine reducer.
- `calculate_auc` (~2408-2460) - the custom trapezoidal temporal-integration AUC implementation.
- `calculate_vegetation_index` (~2106) - `NDVI = normalizedDifference(['B8','B4'])` renamed to `"index"`.
- S2 collection build (~3617) + `SCL_mask` (~3884) - cloud/shadow masking via the Scene Classification Layer.

All logic below is copied/adapted from the legacy plugin; nothing is invented.

In [ ]:
# Cell 2 - Setup
import os
import ee
import pandas as pd
from IPython.display import Image, display

# Service-account auth via the GEE env var (same as dev/testing.ipynb).
credentials = ee.ServiceAccountCredentials(None, os.environ["GEE"])
ee.Initialize(credentials)

print("Earth Engine initialized.")

In [ ]:
# Cell 3 - AOI + dates + masked S2 collection mapped to a single NDVI band over time

# --- AOI (from project shapefile, same area as dev/testing.ipynb) ---
import zipfile

import geopandas as gpd


def load_aoi_from_shapefile(shapefile_path):
    """Load an AOI from a (zipped) shapefile as an ee.FeatureCollection.

    Same loader as dev/testing.ipynb: reproject to EPSG:4326, dissolve to a
    single geometry, strip any Z coordinate, wrap in a FeatureCollection.
    """
    if shapefile_path.endswith(".zip"):
        with zipfile.ZipFile(shapefile_path, "r") as zip_ref:
            shapefile_within_zip = None
            for file in zip_ref.namelist():
                if file.lower().endswith(".shp"):
                    shapefile_within_zip = file
                    break
            if not shapefile_within_zip:
                raise FileNotFoundError(f"No .shp file found inside the zip archive: {shapefile_path}")
            gpd_aoi = gpd.read_file(f"zip://{shapefile_path}/{shapefile_within_zip}")
    else:
        gpd_aoi = gpd.read_file(shapefile_path)

    gpd_aoi = gpd_aoi.to_crs(epsg=4326)

    if gpd_aoi.empty:
        raise ValueError(f"The shapefile at {shapefile_path} does not contain any geometries.")

    if len(gpd_aoi) > 1:
        gpd_aoi = gpd_aoi.dissolve()

    geometry = gpd_aoi.geometry.iloc[0]
    geojson = geometry.__geo_interface__

    if geojson["type"] == "Polygon":
        geojson["coordinates"] = [
            list(map(lambda coord: coord[:2], ring)) for ring in geojson["coordinates"]
        ]
    elif geojson["type"] == "MultiPolygon":
        geojson["coordinates"] = [
            [list(map(lambda coord: coord[:2], ring)) for ring in polygon]
            for polygon in geojson["coordinates"]
        ]

    ee_geometry = ee.Geometry(geojson)
    feature = ee.Feature(ee_geometry)
    return ee.FeatureCollection([feature])


aoi = load_aoi_from_shapefile("contorno_area_total.zip").geometry()

# --- Date range (legacy: self.inicio / self.final) ---
inicio = "2023-01-01"
final = "2023-12-31"
nuvem = 40  # tile cloud percentage limit (legacy: self.nuvem)

# SCL classes to mask out. Legacy SCL_mask keeps pixels NOT in the selected set.
# Here we remove: 3=cloud shadows, 8=cloud medium prob, 9=cloud high prob,
# 10=thin cirrus, 11=snow/ice (mirrors typical legacy checkbox selection).
SCL_CLASSES_TO_MASK = [3, 8, 9, 10, 11]


def scl_mask(image):
    """Legacy SCL_mask: keep only pixels whose SCL value is not in the masked set."""
    scl = image.select("SCL")
    mask = scl.neq(SCL_CLASSES_TO_MASK[0])
    for class_value in SCL_CLASSES_TO_MASK[1:]:
        mask = mask.And(scl.neq(class_value))
    return image.updateMask(mask)


def calculate_ndvi(image):
    """Legacy calculate_vegetation_index for NDVI: normalizedDifference(B8,B4) -> 'index'.
    Preserve system:time_start (legacy copyProperties in composite())."""
    ndvi = image.normalizedDifference(["B8", "B4"]).rename("index")
    return ndvi.copyProperties(image, ["system:time_start"])


# Build the masked Sentinel-2 SR collection (legacy: COPERNICUS/S2_SR_HARMONIZED)
sentinel2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterDate(inicio, final)
    .filterBounds(aoi)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", nuvem))
    .map(lambda image: image.set("date", image.date().format("YYYY-MM-dd")))
    .map(scl_mask)
)

# Map to a single NDVI band per date -> the index collection composited below.
index_collection = ee.ImageCollection(sentinel2.map(calculate_ndvi))

print("S2 images in range:", sentinel2.size().getInfo())
print("NDVI images in index collection:", index_collection.size().getInfo())

In [ ]:
# Cell 4 - Metric -> reducer map, Amplitude image math, and the legacy AUC

# Reducer-based metrics (legacy aggregate_index_collection). The collection
# shortcut methods (.mean()/.median()/...) are the ImageCollection equivalents
# of reducing each pixel across time with these reducers.
METRICS = {
    "Mean": ee.Reducer.mean(),
    "Median": ee.Reducer.median(),
    "Min": ee.Reducer.min(),
    "Max": ee.Reducer.max(),
    "Standard Deviation": ee.Reducer.stdDev(),
    "Sum": ee.Reducer.sum(),
}


def calculate_auc(index_collection, start_date):
    """Area Under Curve via the trapezoidal rule over time.
    Copied from legacy calculate_auc (ravi_dialog.py ~2408-2460)."""
    print("Calculating AUC...")
    count = index_collection.size().getInfo()
    if count < 2:
        raise ValueError("Insufficient number of images to calculate AUC.")

    # Get the first image to use as a spatial reference
    first_image = index_collection.first()

    # Convert collection to multi-band image while preserving projection
    index_stack = index_collection.toBands()

    # Define a valid mask (minimum mask value of all bands)
    valid_mask = index_stack.mask().reduce(ee.Reducer.min())

    # Define the start date
    start_date = ee.Date(start_date)

    # Create a list of timestamps in days relative to the start date
    timestamps = index_collection.aggregate_array("system:time_start").map(
        lambda date: ee.Date(date).difference(start_date, "day")
    )

    # Calculate time differences between consecutive images
    time_diffs = ee.List(timestamps).slice(0, -1).zip(ee.List(timestamps).slice(1)).map(
        lambda pair: ee.Number(ee.List(pair).get(1)).subtract(ee.Number(ee.List(pair).get(0)))
    )

    # Convert the index stack to an array
    index_array = index_stack.toArray()

    # Sums of index values for consecutive images
    sums = index_array.arraySlice(0, 1).add(index_array.arraySlice(0, 0, -1))

    # AUC using the trapezoidal rule
    auc = ee.Image.constant(time_diffs).toArray().multiply(sums).divide(2).arrayReduce(
        ee.Reducer.sum(), [0]
    )

    # Extract result and apply mask
    auc_image = auc.arrayGet([0]).updateMask(valid_mask)

    # New image with the same footprint as the first image
    final_image = first_image.select(0).multiply(0).add(auc_image)
    return final_image


def composite_by_metric(index_collection, metrica, start_date=inicio):
    """Aggregate the index collection into ONE ee.Image by the chosen metric.
    Mirrors legacy aggregate_index_collection metric_functions."""
    first_image = index_collection.first()

    if metrica == "Amplitude":
        result_image = index_collection.max().subtract(index_collection.min())
    elif metrica == "Area Under Curve (AUC)":
        result_image = calculate_auc(index_collection, start_date)
    elif metrica in METRICS:
        result_image = index_collection.reduce(METRICS[metrica])
    else:
        valid = ", ".join(list(METRICS) + ["Amplitude", "Area Under Curve (AUC)"])
        raise ValueError(f"Invalid metric: {metrica}. Valid metrics are: {valid}")

    # Keep the spatial alignment of the first image (legacy setDefaultProjection + clip)
    aligned = result_image.setDefaultProjection(
        first_image.projection()
    ).rename("index")
    return ee.Image(aligned)

In [ ]:
# Cell 5 - Composite each metric and report reduceRegion mean over AOI in a table
ALL_METRICS = list(METRICS) + ["Amplitude", "Area Under Curve (AUC)"]

rows = []
for metrica in ALL_METRICS:
    composite = composite_by_metric(index_collection, metrica)
    stats = composite.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=aoi,
        scale=10,
        bestEffort=True,
    )
    value = stats.get("index").getInfo()
    rows.append({"metric": metrica, "AOI_mean": value})

df = pd.DataFrame(rows).set_index("metric")
df

In [ ]:
# Cell 6 - Render one composite as an inline thumbnail with an NDVI palette
metrica = "Mean"  # try any of ALL_METRICS
composite = composite_by_metric(index_collection, metrica)

# NDVI-style color ramp (brown -> yellow -> green)
palette = [
    "#a50026", "#d73027", "#f46d43", "#fdae61", "#fee08b",
    "#d9ef8b", "#a6d96a", "#66bd63", "#1a9850", "#006837",
]

# Mean/Median/Min/Max/Amplitude/StdDev sit in the NDVI value domain; Sum and AUC
# span much larger ranges, so scale the visualization bounds accordingly.
if metrica == "Sum":
    vis_min, vis_max = 0, 20
elif metrica == "Area Under Curve (AUC)":
    vis_min, vis_max = 0, 300
elif metrica in ("Amplitude", "Standard Deviation"):
    vis_min, vis_max = 0, 1
else:
    vis_min, vis_max = -0.2, 1.0

thumb_url = composite.clip(aoi).getThumbURL({
    "min": vis_min,
    "max": vis_max,
    "palette": palette,
    "region": aoi,
    "dimensions": 512,
    "format": "png",
})

print(f"{metrica} composite thumbnail:")
print(thumb_url)
display(Image(url=thumb_url))